# Over-Fitting and Model Tuning

## Initializations

In [3]:
# Required packages
import pandas as pd       # For data handling
import numpy as np        # For numerical operations (optional but useful)
import matplotlib.pyplot as plt  # For plotting
import seaborn as sns     # For nicer statistical plots
import scienceplots
from scipy.stats import skew
from sklearn.preprocessing import PowerTransformer # For transformations 
from sklearn.decomposition import PCA # For Principal Component Analysis
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, leaves_list

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import KNNImputer
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

import sys
import os

# Absolute path to the repo root
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../"))

if repo_root not in sys.path:
    sys.path.append(repo_root)

from utils import check_transform_suitability, selective_transform, near_zero_var, find_correlation, plot_corr

# Set the plotting style
sns.set_theme(style="darkgrid")  # Set theme for seaborn plots

# Load data required for the analysis in this notebook
two_cd = pd.read_parquet("../../data/twoClassData.parquet")
german_credit = pd.read_parquet("../../data/GermanCredit.parquet")
chem_man_pro = pd.read_parquet("../../data/ChemicalManufacturingProcess.parquet")

## Investigate data

In [12]:
two_cd.head()

,classes,PredictorA,PredictorB
0,Class2,0.1582,0.1609
1,Class2,0.6552,0.4918
2,Class2,0.7060,0.6333
3,Class2,0.1992,0.0881
4,Class2,0.3952,0.4152


In [16]:
print(two_cd["classes"].unique(), "\n")      # List of unique class labels
print(two_cd["classes"].nunique())     # Number of unique classes
print(two_cd["classes"].value_counts()) # Frequency of each class

['Class2', 'Class1']
Categories (2, object): ['Class1', 'Class2'] 

2
classes
Class1    111
Class2     97
Name: count, dtype: int64


In [ ]:
two_cd.describe()   # Summary statistics for numeric columns

,PredictorA,PredictorB
count,208.000000,208.000000
mean,0.250221,0.236013
std,0.140072,0.132705
min,0.023600,0.028900
25%,0.133475,0.129250
50%,0.249050,0.224800
75%,0.331250,0.301650
max,0.706000,0.734200


In [8]:
two_cd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   classes     208 non-null    category
 1   PredictorA  208 non-null    float64 
 2   PredictorB  208 non-null    float64 
dtypes: category(1), float64(2)
memory usage: 3.7 KB


## Data Splitting